# 🤖 Autonomous Multi-Agent DevOps Assistant

A learning-focused DevOps assistant prototype that coordinates specialized agents to inspect a project, review CI/CD configuration, identify operational risks, and produce a prioritized action plan.

**Safety by design:** this notebook performs read-only analysis on sample project data. It does not deploy software, change infrastructure, run shell commands, access cloud accounts, or execute suggested remediations.

## What you'll build

- 🧭 Coordinator agent that orchestrates specialist reviews
- 🔎 Repository hygiene and secret-scan status review
- 🧪 CI/CD checks for tests, linting, security scans, production approvals, and smoke tests
- 🛡️ Security and supply-chain checks
- 📈 Monitoring and rollback-readiness review
- 📝 Severity-ranked report with evidence and recommendations
- 📤 JSON and Markdown report export

This is a transparent rule-based multi-agent simulation. No API key is required; an optional LLM extension is outlined later.

## 1. Imports and sample project

The baseline uses only the Python standard library.

In [ ]:
from dataclasses import dataclass, asdict
from typing import List, Dict, Any
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json

In [ ]:
project = {
    "name": "sample-web-service",
    "language": "Python",
    "runtime": "Python 3.11",
    "dependencies": ["fastapi", "requests"],
    "has_tests": True,
    "has_readme": True,
    "has_dockerfile": True,
    "has_monitoring": False,
    "has_rollback_plan": False,
    "secrets_in_repo": "unknown",
    "deployment_strategy": "direct",
}

pipeline = {
    "runs_tests": True,
    "runs_linter": False,
    "dependency_scan": False,
    "secret_scan": False,
    "uses_pinned_actions": False,
    "requires_approval_for_production": False,
    "has_deployment_smoke_test": False,
}

print("Project:", project["name"])
print("Pipeline configuration:", pipeline)

## 2. Define a shared finding format

Each specialist returns structured findings so the coordinator can combine them consistently.

In [ ]:
@dataclass
class Finding:
    agent: str
    title: str
    severity: str
    category: str
    evidence: str
    recommendation: str
    confidence: str = "medium"
    requires_human_approval: bool = True

SEVERITY_ORDER = {"critical": 0, "high": 1, "medium": 2, "low": 3, "info": 4}

def finding(agent, title, severity, category, evidence, recommendation, confidence="medium"):
    return Finding(agent, title, severity, category, evidence, recommendation, confidence, True)

## 3. Specialist agents

These transparent checks use the supplied sample metadata; they do not inspect a live repository.

In [ ]:
def repository_analyst(project: Dict[str, Any]) -> List[Finding]:
    findings = []
    if not project.get("has_tests"):
        findings.append(finding("Repository Analyst", "No test suite detected", "high", "quality",
            "has_tests is false", "Add unit and integration tests and run them in CI.", "high"))
    if not project.get("has_readme"):
        findings.append(finding("Repository Analyst", "README is missing", "low", "maintainability",
            "has_readme is false", "Document setup, configuration, testing, and deployment.", "high"))
    if not project.get("has_dockerfile"):
        findings.append(finding("Repository Analyst", "Container build definition not detected", "low", "delivery",
            "has_dockerfile is false", "If container deployment is intended, add a reproducible container build.", "medium"))
    if project.get("secrets_in_repo") == "yes":
        findings.append(finding("Repository Analyst", "Potential secrets committed to repository", "critical", "security",
            "secrets_in_repo is yes", "Revoke exposed credentials, rotate them, and move secrets to an approved secret manager.", "medium"))
    elif project.get("secrets_in_repo") == "unknown":
        findings.append(finding("Repository Analyst", "Secret exposure status is unknown", "medium", "security",
            "No verified secret-scan result supplied", "Run an approved secret scanner and review findings; do not print secret values.", "high"))
    return findings

def cicd_reviewer(pipeline: Dict[str, Any]) -> List[Finding]:
    checks = [
        ("runs_tests", "CI does not run tests", "high", "quality", "Add automated tests before progressing a build."),
        ("runs_linter", "CI does not run a linter", "low", "quality", "Add linting or static checks appropriate for the language."),
        ("dependency_scan", "Dependency scanning is not enabled", "medium", "security", "Add dependency vulnerability scanning and a triage policy."),
        ("secret_scan", "Secret scanning is not enabled", "high", "security", "Add secret scanning and a response process."),
        ("uses_pinned_actions", "CI actions are not pinned to immutable references", "medium", "supply-chain", "Pin third-party actions to reviewed immutable commit SHAs where supported."),
        ("requires_approval_for_production", "Production deployment has no approval gate", "high", "change-management", "Add an environment protection rule or equivalent controlled approval gate."),
        ("has_deployment_smoke_test", "No post-deployment smoke test is configured", "medium", "reliability", "Run health checks after deployment and define failure handling."),
    ]
    findings = []
    for key, title, severity, category, recommendation in checks:
        if not pipeline.get(key):
            findings.append(finding("CI/CD Reviewer", title, severity, category,
                f"{key} is false", recommendation, "high"))
    return findings

def observability_reviewer(project: Dict[str, Any]) -> List[Finding]:
    findings = []
    if not project.get("has_monitoring"):
        findings.append(finding("Observability Reviewer", "Monitoring status not configured", "medium", "observability",
            "has_monitoring is false", "Define service indicators, dashboards, logs, and actionable alerts.", "high"))
    if not project.get("has_rollback_plan"):
        findings.append(finding("Observability Reviewer", "Rollback plan not documented", "high", "reliability",
            "has_rollback_plan is false", "Document and test a rollback or roll-forward procedure before production release.", "high"))
    if project.get("deployment_strategy") == "direct":
        findings.append(finding("Observability Reviewer", "Deployment strategy is direct", "medium", "release-safety",
            "deployment_strategy is direct", "Consider staged, canary, or blue-green releases when appropriate.", "medium"))
    return findings

def security_reviewer(project: Dict[str, Any], pipeline: Dict[str, Any]) -> List[Finding]:
    findings = []
    if not pipeline.get("secret_scan"):
        findings.append(finding("Security Reviewer", "Pipeline lacks secret scanning", "high", "security",
            "secret_scan is false", "Add secret detection and a documented response process.", "high"))
    if not pipeline.get("dependency_scan"):
        findings.append(finding("Security Reviewer", "Pipeline lacks dependency vulnerability checks", "medium", "security",
            "dependency_scan is false", "Scan dependencies and review remediation priority before release.", "high"))
    if not pipeline.get("requires_approval_for_production"):
        findings.append(finding("Security Reviewer", "No production approval gate detected", "high", "change-management",
            "requires_approval_for_production is false", "Require authorized review for production changes.", "high"))
    return findings

## 4. Coordinator agent

The coordinator runs each specialist, combines findings, removes duplicates by title, and sorts by severity.

In [ ]:
def coordinate_review(project: Dict[str, Any], pipeline: Dict[str, Any]) -> List[Finding]:
    agent_results = [
        repository_analyst(project),
        cicd_reviewer(pipeline),
        observability_reviewer(project),
        security_reviewer(project, pipeline),
    ]
    combined = [item for result in agent_results for item in result]
    unique = {}
    for item in combined:
        unique.setdefault(item.title.strip().lower(), item)
    return sorted(unique.values(), key=lambda x: (SEVERITY_ORDER.get(x.severity, 99), x.title.lower()))

findings = coordinate_review(project, pipeline)
print(f"Specialist agents: 4 | Unique findings: {len(findings)}")

## 5. Build a prioritized report

In [ ]:
def build_report(findings: List[Finding]) -> Dict[str, Any]:
    severity_counts = Counter(item.severity for item in findings)
    return {
        "project": project.get("name", "unknown"),
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "mode": "read-only-demo",
        "summary": {
            "total_findings": len(findings),
            "by_severity": dict(severity_counts),
            "agents": sorted({item.agent for item in findings}),
        },
        "findings": [asdict(item) for item in findings],
        "safety_note": "Recommendations only. Verify evidence and obtain authorized human approval before making changes.",
    }

report = build_report(findings)
print(json.dumps(report["summary"], indent=2))

## 6. Review findings

Severity is a triage suggestion, not a formal risk rating.

In [ ]:
for item in findings:
    print(f"\\n[{item.severity.upper()}] {item.title} — {item.agent}")
    print("  Evidence:", item.evidence)
    print("  Recommendation:", item.recommendation)
    print("  Human approval required:", item.requires_human_approval)

## 7. Export JSON and Markdown reports

In [ ]:
def export_reports(report: Dict[str, Any], output_dir="."):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    json_path = output_dir / "devops_review_report.json"
    md_path = output_dir / "devops_review_report.md"
    json_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

    lines = [
        f"# DevOps Review Report: {report['project']}",
        "",
        f"- Generated (UTC): {report['generated_at_utc']}",
        f"- Mode: {report['mode']}",
        f"- Total findings: {report['summary']['total_findings']}",
        "",
        "## Findings",
    ]
    for item in report["findings"]:
        lines.extend([
            "",
            f"### [{item['severity'].upper()}] {item['title']}",
            f"- **Agent:** {item['agent']}",
            f"- **Category:** {item['category']}",
            f"- **Evidence:** {item['evidence']}",
            f"- **Recommendation:** {item['recommendation']}",
            f"- **Confidence:** {item['confidence']}",
            f"- **Human approval required:** {item['requires_human_approval']}",
        ])
    lines.extend(["", f"> {report['safety_note']}", ""])
    md_path.write_text("\n".join(lines), encoding="utf-8")
    return json_path, md_path

json_report, markdown_report = export_reports(report)
print("JSON report:", json_report.resolve())
print("Markdown report:", markdown_report.resolve())

## 8. Optional LLM upgrade

You can replace rule-based specialists with LLM-backed agents using LangGraph, CrewAI, or the OpenAI Agents SDK. Keep agents narrowly scoped and require schema-validated output.

Recommended guardrails:
- Read-only and least-privilege access by default
- Input/output validation and bounded retries
- Evidence required for findings; distinguish facts from assumptions
- Secret redaction and environment-based credentials
- No arbitrary shell commands generated or executed by the model
- Human approval for deployments, infrastructure changes, permission changes, or destructive actions
- Audit logs, dry-run plans, and rollback procedures

## 9. Roadmap

- [ ] Parse GitHub Actions YAML workflows
- [ ] Add Dockerfile and dependency-manifest analysis
- [ ] Add read-only inspection of a user-selected repository
- [ ] Add tests for each rule and report schema
- [ ] Add optional LLM agents with structured output
- [ ] Add an approval queue and audit trail
- [ ] Add a dry-run remediation planner
- [ ] Add authorized observability integrations
- [ ] Add a Streamlit interface